# BARAM 2026 — 최종 파이프라인 (v9 = v7 + 시드 10)

**v6(LB 0.6292) + 2폴드 검증 채택분 2건:**
1. **SCADA 라벨 정제** — 원본 SCADA로 "바람 있는데 발전≈0"(고장·출력제한) 시간 식별 → 학습 가중 0.5 (두 폴드 +0.008~0.011, `scada_clean_result.json`)
2. **GBM 튜닝** — lr 0.0207·트리 2000·min_child 300·colsample 0.5 (두 폴드 +0.003~0.004, `gbm_hpo_result.json`)

유지: feature 65 · GBM⊕MLP(시드5) 블렌드 w=0.7 · **보수적 nudge**(scale≤1.05, shift≤2%; v6에서 실전 검증 — LB↔홀드아웃 편차 +0.002).

구성: §0 설정·SCADA → §1 **결합 검증**(2023 폴드, 시드3) → §2 holdout(2024, 시드5) + nudge fit → §3 전체 재학습 → `submission_v9.csv`.

> ⚠️ macOS torch+LightGBM OMP 가드 필수(첫 셀). SCADA는 학습 가중치 산출에만 사용(테스트 입력 아님 — 규칙 적합).

## 0. 설정 + SCADA stuck 마스크

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores, metric

DEV="mps" if torch.backends.mps.is_available() else "cpu"
SEEDS=[15,0,1,2,3,4,5,6,7,8]; BLEND=0.7; STUCK_W=0.5
MLP_CFG=dict(h=256,depth=3,drop=0.0,lr=0.0015868563457489381,wd=1e-4,bs=256,emb=4)
GBM_PARAMS=dict(objective="mae",n_estimators=2000,learning_rate=0.020651822836313095,
    num_leaves=63,min_child_samples=300,subsample=0.8,subsample_freq=1,colsample_bytree=0.5,
    reg_lambda=0.1,random_state=42,n_jobs=1,verbose=-1)     # gbm_hpo_result.json trial7
RAW=os.environ.get("WIND_RAW_DIR",os.path.expanduser("~/Downloads/open"))
GROUPS=(1,2,3)
print(f"device={DEV} seeds={SEEDS} blend={BLEND} stuck_w={STUCK_W}")

FR,TGT,FR_TE={},{},{}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
    FR_TE[g]=W.add_spatial(W.build(W.load_test(g),g),"test")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
FEATS=W.lean_features(BASE_ALL)+W.SPATIAL_COLS

# SCADA → 그룹-시간 stuck 마스크 (풍속>=4 m/s & 출력<=정격1%, 보고터빈>=3, 비율>=30%)
def stuck_table():
    frames={}
    for fn,pre,n,rate in [("scada_vestas_train.csv","vestas",12,3600.0),
                          ("scada_unison_train.csv","unison",5,4200.0)]:
        d=pd.read_csv(f"{RAW}/train/{fn}",encoding="utf-8-sig",parse_dates=["kst_dtm"])
        d["hour"]=d["kst_dtm"].dt.ceil("h")
        agg={}
        for i in range(1,n+1):
            pw=f"{pre}_wtg{i:02d}_power_kw10m"; ws=f"{pre}_wtg{i:02d}_ws"
            h=d.groupby("hour").agg(pw_m=(pw,"mean"),ws_m=(ws,"mean"))
            agg[i]=pd.DataFrame({f"stuck_{i}":((h.ws_m>=4.0)&(h.pw_m<=0.01*rate)).astype(float),
                                 f"rep_{i}":h.pw_m.notna().astype(float)})
        frames[pre]=pd.concat(agg.values(),axis=1)
    def frac(pre,ids):
        f=frames[pre]
        st=f[[f"stuck_{i}" for i in ids]].sum(axis=1); rp=f[[f"rep_{i}" for i in ids]].sum(axis=1)
        return (st/rp).where(rp>=3)
    return {1:frac("vestas",range(1,7)),2:frac("vestas",range(7,13)),3:frac("unison",range(1,6))}
STUCK=stuck_table()
for g in GROUPS:
    s=STUCK[g].reindex(FR[g].kst_dtm).to_numpy()
    FR[g]=FR[g].assign(stuck_mask=(np.nan_to_num(s,nan=0.0)>=0.3))
    print(f"g{g}: stuck 마스크 {int(FR[g].stuck_mask.sum())}시간 ({100*FR[g].stuck_mask.mean():.1f}%) → 가중 {STUCK_W}")

device=mps seeds=[15, 0, 1, 2, 3, 4, 5, 6, 7, 8] blend=0.7 stuck_w=0.5


g1: stuck 마스크 4727시간 (18.0%) → 가중 0.5
g2: stuck 마스크 4872시간 (18.6%) → 가중 0.5
g3: stuck 마스크 778시간 (4.4%) → 가중 0.5


## 0b. 모델 정의 (가중 학습)

In [2]:
class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=256,depth=3,drop=0.0,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        L=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1): L+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        L+=[nn.Linear(h,1)]; s.net=nn.Sequential(*L)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)

def train_one(pool_tr,seed):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[FEATS].mean(); sd=pool_tr[FEATS].std()+1e-8
    X=((pool_tr[FEATS]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    wt=pool_tr["w"].to_numpy(np.float32)
    n=len(X); cut=int(n*0.85)
    Xt=torch.tensor(X,device=DEV); yt=torch.tensor(y,device=DEV)
    gt=torch.tensor(gid,device=DEV); wtt=torch.tensor(wt,device=DEV)
    net=MLP(len(FEATS),**{k:MLP_CFG[k] for k in ("h","depth","drop","emb")}).to(DEV)
    opt=torch.optim.AdamW(net.parameters(),lr=MLP_CFG["lr"],weight_decay=MLP_CFG["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),MLP_CFG["bs"]):
            b=torch.tensor(perm[i:i+MLP_CFG["bs"]],device=DEV); opt.zero_grad()
            e=(net(Xt[b],gt[b])-yt[b]).abs()
            ((e*wtt[b]).sum()/(wtt[b].sum()+1e-8)).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            e=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs()
            vl=((e*wtt[cut:]).sum()/(wtt[cut:].sum()+1e-8)).item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def predict_one(net,scaler,fr,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[FEATS]-mu)/sd).to_numpy(np.float32),device=DEV)
    gid=torch.full((len(fr),),g-1,dtype=torch.long,device=DEV)
    net.eval()
    with torch.no_grad(): p=net(X,gid).cpu().numpy()
    return np.clip(p,0,1)*cap

def blend_predict(tr_frames, seeds=None):
    seeds=seeds or SEEDS
    pool=[]
    for g,(tr2,_) in tr_frames.items():
        p=tr2[FEATS+["kst_dtm","stuck_mask"]].copy()
        p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1
        p["w"]=np.where(p["stuck_mask"],STUCK_W,1.0); pool.append(p)
    pool=pd.concat(pool,ignore_index=True)
    acc={g:[] for g in tr_frames}
    for sd_ in seeds:
        net,scaler=train_one(pool,sd_)
        for g,(_,va2) in tr_frames.items():
            acc[g].append(predict_one(net,scaler,va2,g,W.CAP[g]))
    out={}
    for g,(tr2,va2) in tr_frames.items():
        cap=W.CAP[g]; tgt=TGT[g]
        w=np.where(tr2["stuck_mask"],STUCK_W,1.0)
        lg_=lgb.LGBMRegressor(**GBM_PARAMS).fit(tr2[FEATS],tr2[tgt],sample_weight=w)
        hg_=histgbm().fit(tr2[FEATS].to_numpy(),tr2[tgt].to_numpy(),sample_weight=w)
        gp=np.clip(0.5*(lg_.predict(va2[FEATS])+hg_.predict(va2[FEATS].to_numpy())),0,cap)
        out[g]=np.clip((1-BLEND)*gp+BLEND*np.mean(acc[g],axis=0),0,cap)
    return out

## 1. 결합 검증 — 2023 폴드 (시드3, raw blend)

SCADA와 GBM 튜닝은 **개별** 채택이므로 결합이 손해가 아닌지 확인. 기준: SCADA만 적용한 raw 0.6098 이상.

In [3]:
ent={}
for g in (1,2):     # 2022→2023 폴드 (g3는 2022 없음)
    tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
    tr=fr[yr==2022]; va=fr[yr==2023]
    iso=W.fit_powercurve(tr,tgt,cap)
    ent[g]=(W.with_pc(tr,iso),W.with_pc(va,iso))
pred23=blend_predict(ent,seeds=[15,0,1])
nm=[]; fi=[]
for g,(_,va2) in ent.items():
    a,b=group_scores(va2[TGT[g]].to_numpy(),pred23[g],W.CAP[g]); nm.append(a); fi.append(b)
t23=0.5*(1-np.mean(nm))+0.5*np.mean(fi)
print(f"결합(SCADA 0.5 + 튜닝GBM) 2023 폴드 raw = {t23:.4f}")
print("참조: v6구성 raw=0.5986, SCADA만=0.6098 — 이 값 이상이면 결합 무해 확인")
assert t23>=0.6, f"결합 검증 실패({t23:.4f}) — 채택 재검토 필요"

결합(SCADA 0.5 + 튜닝GBM) 2023 폴드 raw = 0.6098
참조: v6구성 raw=0.5986, SCADA만=0.6098 — 이 값 이상이면 결합 무해 확인


## 2. 2024 holdout (시드5) → 보수적 nudge fit

In [4]:
tr_frames={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]
    m_tr=fr.kst_dtm<W.VALID_START; m_va=fr.kst_dtm>=W.VALID_START
    iso=W.fit_powercurve(fr[m_tr],tgt,cap)
    tr_frames[g]=(W.with_pc(fr[m_tr],iso),W.with_pc(fr[m_va],iso))
val_pred=blend_predict(tr_frames)
print("holdout 예측 완료")

OOF={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]; tr2=tr_frames[g][0]
    oof=np.full(len(tr2),np.nan); yrs=sorted(tr2.kst_dtm.dt.year.unique())
    if len(yrs)>=2:
        for ty in yrs:
            m_in=tr2.kst_dtm.dt.year!=ty; m_out=(tr2.kst_dtm.dt.year==ty).to_numpy()
            oof[m_out]=blend_predict({g:(tr2[m_in],tr2[tr2.kst_dtm.dt.year==ty])})[g]
    else:
        n=len(tr2); cut=int(n*0.7)
        oof[cut:]=blend_predict({g:(tr2.iloc[:cut],tr2.iloc[cut:])})[g]
    OOF[g]=oof
    print(f"g{g} OOF {int(np.isfinite(oof).sum())}/{len(tr2)}")

def nudge_fit(tr,tgt,oof,cap,smax=1.05,shmax=0.02):
    d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])]
    yt=d[tgt].to_numpy(); yp=d["oof"].to_numpy(); best=(1.0,0.0); bf=-1
    for s in np.linspace(2-smax,smax,21):
        for sh in np.linspace(-shmax,shmax,21)*cap:
            f=group_scores(yt,np.clip(yp*s+sh,0,cap),cap)[1]
            if f>bf: bf=f; best=(s,sh)
    return best

STORE={}
for g in GROUPS:
    STORE[g]=nudge_fit(tr_frames[g][0],TGT[g],OOF[g],W.CAP[g])
    print(f"g{g}: nudge scale={STORE[g][0]:.3f} shift={STORE[g][1]:+.0f}")
def apply_post(g,pred):
    cap=W.CAP[g]; s,sh=STORE[g]; return np.clip(pred*s+sh,0,cap)

ans=pd.DataFrame({f"kpx_group_{g}":tr_frames[g][1][TGT[g]].to_numpy() for g in GROUPS})
p_post=pd.DataFrame({f"kpx_group_{g}":apply_post(g,val_pred[g]) for g in GROUPS})
ts,omn,fi=metric(ans,p_post)
print(f"\n2024 holdout(참고) = {ts:.4f} (1-NMAE {omn:.4f}, FICR {fi:.4f})")
print("v7 동일지표(시드5) 참조: 0.6351 (LB 실측 0.63497). 시드10 효과는 +0.001 수준 기대")

holdout 예측 완료


g1 OOF 17421/17421


g2 OOF 17422/17422


g3 OOF 2628/8759
g1: nudge scale=1.050 shift=+432
g2: nudge scale=1.025 shift=+432
g3: nudge scale=1.045 shift=+378

2024 holdout(참고) = 0.6364 (1-NMAE 0.8755, FICR 0.3973)
v7 동일지표(시드5) 참조: 0.6351 (LB 실측 0.63497). 시드10 효과는 +0.001 수준 기대


## 3. 전체 재학습 → submission_v9.csv

In [5]:
full_frames={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]
    iso=W.fit_powercurve(FR[g],tgt,cap)
    full_frames[g]=(W.with_pc(FR[g],iso),W.with_pc(FR_TE[g],iso))
test_pred=blend_predict(full_frames)
out=W.load_test(1)[["forecast_id","kst_dtm"]].rename(columns={"kst_dtm":"forecast_kst_dtm"})
for g in GROUPS:
    out[f"kpx_group_{g}"]=apply_post(g,test_pred[g])
assert out.shape[0]==8760
for g in GROUPS:
    c=out[f"kpx_group_{g}"]; assert (c>=0).all() and (c<=W.CAP[g]).all() and c.notna().all()
out.to_csv("submission/ver_9/submission.csv",index=False); print("saved submission/ver_9/submission.csv",out.shape)
bias={g:round(100*out[f"kpx_group_{g}"].mean()/W.CAP[g],1) for g in GROUPS}
print("평균 이용률(%):",bias)
assert max(bias.values())<=39.5, f"편향 가드 위반: {bias}"   # v5·v8 교훈

saved submission_v9.csv (8760, 5)
평균 이용률(%): {1: np.float64(38.0), 2: np.float64(39.4), 3: np.float64(33.4)}


## 결과 메모
- v7 = v6 + SCADA 라벨정제(w0.5) + 튜닝 GBM. 캘리브레이션 근거(v6: holdout 0.6273 ↔ LB 0.6292)로 §2의 holdout 값 ≈ LB 기대치.
- 채택 근거: `scada_clean_result.json`, `gbm_hpo_result.json` (각각 2폴드 모두 우위).